# CLEAR pulse





In [ ]:
import Pkg
Pkg.add(["DifferentialEquations", "Plots", "LaTeXStrings", "LsqFit", "JSON"])

In [ ]:
using DifferentialEquations
using Plots
using LaTeXStrings
using TOML

project_dir = pwd()
include(joinpath(project_dir, "funz", "CLEAR_Reset.jl"))

using .CLEAR_Reset

# 1. Parameters


In [ ]:
par = TOML.parsefile(joinpath(project_dir, "parametri", "risonatore.toml"))

kappa = par["kappa"]
alpha0 = par["alpha0_real"] + im * par["alpha0_imag"]
t_kick = par["t_kick"]
t_readout = par["t_readout"]
n_target = par["n_target"]
f_res_g_MHz = par["f_res_g_MHz"]
f_res_e_MHz = par["f_res_e_MHz"]
f_drive_MHz = par["f_drive_MHz"]

# Coefficiente angolare k del fit lineare n_photons = k * amp^2 + q,
# calcolato in stark_analysis.py a partire dai dati di Stark shift
# e salvato in data/stark_fit_results.json.
import Pkg
Pkg.add(["JSON"])
using JSON

stark_fit_path = joinpath(project_dir, "data", "stark_fit_results.json")
stark_fit = JSON.parsefile(stark_fit_path)
k = stark_fit["k"]
println("k (da fit Stark shift) = ", k)

A_readout = sqrt(n_target/k)
f_mid_MHz = (f_res_g_MHz + f_res_e_MHz) / 2

detuning = 2π * (f_mid_MHz - f_drive_MHz)
chi = π * (f_res_e_MHz - f_res_g_MHz)

println("f_mid / 2π = ", f_mid_MHz, " MHz")
println("detuning = ", detuning, " μs⁻¹")
println("chi = ", chi, " μs⁻¹")
println("|chi| / 2π = ", abs(chi) / (2π), " MHz")

params = ResonatorParams(kappa, detuning; alpha0=alpha0, chi=chi)

println("Parameters:")
println("kappa = ", kappa)
println("chi = ", chi)
println("t_kick = ", t_kick)
println("t_readout = ", t_readout)
println("A_readout = ", A_readout)


# 2. CLEAR pulse amplitudes


In [ ]:
comparison = compare_rect_CLEAR(
    params;
    A=A_readout,
    t_kick=t_kick,
    t_readout=t_readout,
    saveat_points=2000,
)

println("\nCLEAR amplitudes computed from the linear model")
println("ringup1   = ", comparison.kicks.ringup1)
println("ringup2   = ", comparison.kicks.ringup2)
println("readout   = ", comparison.kicks.readout)
println("ringdown1 = ", comparison.kicks.ringdown1)
println("ringdown2 = ", comparison.kicks.ringdown2)

println("\nTime spans")
println("t_rect_stop = ", comparison.t_rect_stop)
println("t_total     = ", comparison.t_total)


# 4. Graph


In [ ]:
plt_pulse_clear = plot_epsilon_clear(
    params,
    t_kick,
    t_readout,
    A_readout,
)

display(plt_pulse_clear)




# 5. Fit della traccia di Ramsey (Fig. 2(a) dell'articolo)

Riproduciamo la Fig. 2(a) di McClure et al., *Rapid Driven Reset of a Qubit Readout Resonator*, Phys. Rev. Applied 5, 011001 (2016): una traccia di Ramsey campione con il fit basato sull'Eq. (1):

$$S(t_R) = \frac{1}{2}\Big[1 - \mathrm{Im}\big\{\exp[-(\Gamma_2 + i\Delta) t_R + i(\varphi_0 - 2 n_0 \chi \tau)]\big\}\Big], \qquad \tau(t_R) = \frac{1 - e^{-(\kappa + 2i\chi) t_R}}{\kappa + 2i\chi}$$

dove $\Delta$ è il detuning di Ramsey, $\Gamma_2 = 1/T_2^{\mathrm{echo}}$ il tasso di decoerenza, $\varphi_0$ la fase iniziale e $n_0$ la popolazione residua del risonatore all'inizio della sequenza di Ramsey. Come nell'articolo, $\kappa$, $\chi$, $\Delta$ e $\Gamma_2$ sono tenuti fissi e gli unici parametri liberi del fit sono $n_0$ e $\varphi_0$.

**Nota:** questo progetto non contiene ancora dati sperimentali di Ramsey. Per mostrare l'intera pipeline (dati -> fit -> grafico) generiamo qui un set di dati sintetico a partire dallo stesso modello (Eq. 1), con parametri "veri" noti ($n_0$, $\varphi_0$) e rumore gaussiano aggiunto; il fit viene poi usato per ristimarli, esattamente come nell'articolo. Quando saranno disponibili le misure reali, basta sostituire `tR_data` e `S_data` con i dati sperimentali (Ramsey delay in μs e ampiezza normalizzata) e rilanciare le celle sottostanti.

In [ ]:
# Parametri della misura di Ramsey.
# kappa e chi sono gia' definiti sopra (Sezione 1, dai parametri del risonatore).
# Delta_Ramsey: detuning del tono di Ramsey (nell'articolo: 10 MHz).
# T2_echo: tempo di decoerenza da cui Gamma2 = 1/T2_echo
#          (valore di esempio: sostituire con la misura di calibrazione Techo_2).

Delta_Ramsey_MHz = 10.0
Delta_Ramsey = 2π * Delta_Ramsey_MHz    # μs^-1

T2_echo = 10.0                          # μs (valore di esempio)
Gamma2 = 1 / T2_echo                    # μs^-1

# Parametri "veri" usati solo per generare la traccia sintetica di esempio
# (n0 ≈ 0.9 come nell'esempio della Fig. 2(a) dell'articolo).
n0_true = 0.9
phi0_true = 0.3

tR_data = range(0.0, 0.6, length=60)    # μs (0-600 ns, come in Fig. 2(a))

S_data = synthetic_ramsey_data(
    tR_data;
    kappa=kappa,
    chi=chi,
    Delta=Delta_Ramsey,
    Gamma2=Gamma2,
    phi0=phi0_true,
    n0=n0_true,
    noise=0.03,
)

println("Parametri usati per generare i dati sintetici:")
println("kappa = ", kappa, "  chi = ", chi)
println("Delta_Ramsey = ", Delta_Ramsey, "  Gamma2 = ", Gamma2)
println("n0_true = ", n0_true, "  phi0_true = ", phi0_true)


In [ ]:
result = fit_ramsey(
    tR_data, S_data;
    kappa=kappa,
    chi=chi,
    Delta=Delta_Ramsey,
    Gamma2=Gamma2,
    p0=[0.5, 0.0],
)

println("\nFit dell'Eq. (1) alla traccia di Ramsey")
println("n0_fit   = ", result.n0, " ± ", result.n0_err)
println("phi0_fit = ", result.phi0, " ± ", result.phi0_err)

tR_fit = range(0.0, 0.6, length=400)
S_fit = ramsey_signal(
    tR_fit;
    kappa=kappa,
    chi=chi,
    Delta=Delta_Ramsey,
    Gamma2=Gamma2,
    phi0=result.phi0,
    n0=result.n0,
)

plt_ramsey = plot_ramsey_fit(
    tR_data, S_data, tR_fit, S_fit;
    title=L"\mathrm{Fig.\ 2(a):\ Ramsey\ experiment\ and\ fit}",
)

display(plt_ramsey)


# 6. Esportazione dei parametri CLEAR per qibolab (Fig. 3(c))

Per riprodurre l'esperimento di Fig. 3(c) (n0 vs potenza di pilotaggio, per impulso quadrato e CLEAR) in `CLEAR_qibolab.py`, esportiamo qui i **rapporti** di ampiezza di ciascun segmento CLEAR rispetto al segmento di readout (`ringup_i / A_readout`, `ringdown_i / A_readout`).

Usiamo i rapporti, non le ampiezze assolute, perché il modello lineare implica che tutte le ampiezze dei kick scalano linearmente con `A_readout`: cosi' i rapporti sono indipendenti dalle unita' (arbitrarie, in μs^-1) usate qui, e in Python bastera' moltiplicarli per l'ampiezza hardware del segmento di readout (che dipende dalla potenza normalizzata P_norm) per ottenere le ampiezze dei 5 segmenti da caricare in `ClearPulseParameters`.

In [ ]:
import Pkg
Pkg.add(["JSON"])
using JSON

kicks_ref = clear_kick_amplitudes(
    kappa=params.kappa,
    detuning=params.detuning,
    chi=params.chi,
    A=A_readout,
    t_kick=t_kick,
)

ratios = Dict(
    "ringup_1"   => [real(kicks_ref.ringup1   / A_readout), imag(kicks_ref.ringup1   / A_readout)],
    "ringup_2"   => [real(kicks_ref.ringup2   / A_readout), imag(kicks_ref.ringup2   / A_readout)],
    "steady"     => [real(kicks_ref.readout   / A_readout), imag(kicks_ref.readout   / A_readout)],
    "ringdown_1" => [real(kicks_ref.ringdown1 / A_readout), imag(kicks_ref.ringdown1 / A_readout)],
    "ringdown_2" => [real(kicks_ref.ringdown2 / A_readout), imag(kicks_ref.ringdown2 / A_readout)],
)

export_data = Dict(
    "t_kick_us"    => t_kick,
    "t_readout_us" => t_readout,
    "ratios"       => ratios,
)

export_path = joinpath(project_dir, "parametri", "clear_amplitudes.json")
open(export_path, "w") do io
    JSON.print(io, export_data, 4)
end

println("Rapporti CLEAR (kick / A_readout):")
for (k, v) in ratios
    println("  ", k, " = ", v[1], " + ", v[2], "im")
end
println("\nSalvato in: ", export_path)
